In [1]:
# # Phase 05.02 — Modality Inference

# ## Objective

# Generate frozen 128-dimensional modality-specific representations
# from raw/demo inputs for the three ASD modalities:

# 1. Genetic
# 2. Behavioral
# 3. Environmental

# Pipeline:

# Raw Input
#    ↓
# Original Modality Preprocessing
#    ↓
# Frozen Representation Model
#    ↓
# 128-D Modality Embedding

# Important:
# - No model training is performed.
# - No checkpoint is modified.
# - No 128→32 projection is performed in this notebook.
# - No multimodal fusion is performed in this notebook.

# The 128→32 common-dimensional projection is handled in Phase 05.03.

In [2]:
# ============================================================
# CELL 2 — IMPORTS AND PROJECT PATHS
# ============================================================

from pathlib import Path
import json
import subprocess
import tempfile

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

import joblib


# ------------------------------------------------------------
# Reproducibility / device
# ------------------------------------------------------------

SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")


# ------------------------------------------------------------
# Project root
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    r"C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening"
)


# ------------------------------------------------------------
# Phase 05 directories
# ------------------------------------------------------------

PHASE05_ROOT = (
    PROJECT_ROOT
    / "06_cross_modal_learning"
    / "05_multimodal_integration"
)

REPORT_DIR = PHASE05_ROOT / "reports" / "05_02_modality_inference"
REPORT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Modality directories
# ------------------------------------------------------------

GENETIC_DIR = PROJECT_ROOT / "Genetic_Model"
BEHAVIORAL_DIR = PROJECT_ROOT / "Behavioral_Model"
ENVIRONMENTAL_DIR = PROJECT_ROOT / "Environmental_Model"


# ------------------------------------------------------------
# Frozen representation checkpoints
# ------------------------------------------------------------

GENETIC_MODEL_PATH = (
    GENETIC_DIR
    / "03_representation_learning"
    / "genetic_transformer_128d.pt"
)

BEHAVIORAL_MODEL_PATH = (
    BEHAVIORAL_DIR
    / "03_representation_learning"
    / "behavioral_encoder_128d.pt"
)

ENVIRONMENTAL_MODEL_PATH = (
    ENVIRONMENTAL_DIR
    / "03_representation_learning"
    / "environmental_encoder_128d.pt"
)


# ------------------------------------------------------------
# Preprocessing artifacts
# ------------------------------------------------------------

BEHAVIORAL_PREPROCESSOR_PATH = (
    BEHAVIORAL_DIR
    / "02_preprocessing"
    / "behavioral_preprocessor.joblib"
)

ENVIRONMENTAL_PREPROCESSOR_PATH = (
    ENVIRONMENTAL_DIR
    / "02_preprocessing"
    / "environmental_preprocessor.joblib"
)


# ------------------------------------------------------------
# Compatibility Python environment for Behavioral preprocessing
# ------------------------------------------------------------

BEHAVIORAL_PYTHON = (
    PROJECT_ROOT
    / ".venv_behavioral_161"
    / "Scripts"
    / "python.exe"
)


# ------------------------------------------------------------
# Demo patient inputs
# ------------------------------------------------------------

DEMO_ROOT = (
    PROJECT_ROOT
    / "05_multimodal_fusion"
    / "M4_demo_patients"
)

DEMO_PATIENTS = [
    "P001",
    "P002",
    "P003",
    "P004",
    "P005",
]


print("Project root:")
print(PROJECT_ROOT)

print("\nPhase 05 report directory:")
print(REPORT_DIR)

print("\nDevice:", DEVICE)
print("Seed:", SEED)

Project root:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening

Phase 05 report directory:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\05_multimodal_integration\reports\05_02_modality_inference

Device: cpu
Seed: 42


In [3]:
# ============================================================
# CELL 3 — VERIFY REQUIRED ARTIFACTS
# ============================================================

required_files = {
    "Genetic representation model": GENETIC_MODEL_PATH,
    "Behavioral representation model": BEHAVIORAL_MODEL_PATH,
    "Environmental representation model": ENVIRONMENTAL_MODEL_PATH,
    "Behavioral preprocessor": BEHAVIORAL_PREPROCESSOR_PATH,
    "Environmental preprocessor": ENVIRONMENTAL_PREPROCESSOR_PATH,
    "Behavioral compatibility Python": BEHAVIORAL_PYTHON,
}


print("=" * 70)
print("REQUIRED ARTIFACT VERIFICATION")
print("=" * 70)

missing = []

for name, path in required_files.items():

    exists = path.exists()

    print(
        f"{name:<35} : "
        f"{'FOUND' if exists else 'MISSING'}"
    )

    if not exists:
        missing.append(str(path))


if missing:
    raise FileNotFoundError(
        "Required artifacts are missing:\n"
        + "\n".join(missing)
    )

print("\n[PASSED] All required artifacts are available.")

REQUIRED ARTIFACT VERIFICATION
Genetic representation model        : FOUND
Behavioral representation model     : FOUND
Environmental representation model  : FOUND
Behavioral preprocessor             : FOUND
Environmental preprocessor          : FOUND
Behavioral compatibility Python     : FOUND

[PASSED] All required artifacts are available.


In [4]:
# ============================================================
# CELL 4 — VERIFY DEMO PATIENT INPUTS
# ============================================================

print("=" * 70)
print("DEMO PATIENT INPUT VERIFICATION")
print("=" * 70)

demo_files = {}

for patient_id in DEMO_PATIENTS:

    patient_dir = DEMO_ROOT / patient_id

    genetic_file = patient_dir / "genetic_input.csv"
    behavioral_file = patient_dir / "behavioral_input.csv"
    environmental_file = patient_dir / "environmental_input.csv"

    demo_files[patient_id] = {
        "genetic": genetic_file,
        "behavioral": behavioral_file,
        "environmental": environmental_file,
    }

    print(f"\n{patient_id}")

    for modality, path in demo_files[patient_id].items():

        status = "FOUND" if path.exists() else "MISSING"

        print(
            f"  {modality:<15}: {status}"
        )

        if not path.exists():
            raise FileNotFoundError(
                f"Missing {modality} input for {patient_id}:\n{path}"
            )


print("\n[PASSED] All demo modality inputs resolved.")

DEMO PATIENT INPUT VERIFICATION

P001
  genetic        : FOUND
  behavioral     : FOUND
  environmental  : FOUND

P002
  genetic        : FOUND
  behavioral     : FOUND
  environmental  : FOUND

P003
  genetic        : FOUND
  behavioral     : FOUND
  environmental  : FOUND

P004
  genetic        : FOUND
  behavioral     : FOUND
  environmental  : FOUND

P005
  genetic        : FOUND
  behavioral     : FOUND
  environmental  : FOUND

[PASSED] All demo modality inputs resolved.


In [5]:
# ============================================================
# CELL 5 — EXACT GENETIC TRANSFORMER
# ============================================================

class GeneticTransformer(nn.Module):
    """
    Frozen Genetic representation model.

    Architecture:

        1000
         ↓
        Linear(1 → 128) applied to each feature
         ↓
        Transformer Encoder
         ↓
        Mean pooling across 1000 features
         ↓
        Linear(128 → 128)
         ↓
        LayerNorm(128)
         ↓
        128-D embedding
    """

    def __init__(self):

        super().__init__()

        # Each scalar genetic feature becomes
        # a 128-dimensional token representation.
        self.feature_embedding = nn.Linear(
            1,
            128
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=128,
            nhead=8,
            dim_feedforward=256,
            batch_first=True,
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2,
        )

        self.output_projection = nn.Sequential(
            nn.Linear(128, 128),
            nn.LayerNorm(128),
        )

    def forward(self, x):

        # Input:
        # [batch, 1000]

        x = x.unsqueeze(-1)

        # [batch, 1000, 1]

        x = self.feature_embedding(x)

        # [batch, 1000, 128]

        x = self.transformer(x)

        # Mean pooling over the 1000 genetic features.
        x = x.mean(dim=1)

        # [batch, 128]

        x = self.output_projection(x)

        return x


print("GeneticTransformer class defined.")

GeneticTransformer class defined.


In [8]:
# ============================================================
# CELL 6 — EXACT BEHAVIORAL + ENVIRONMENTAL ENCODERS
# ============================================================

class BehavioralEncoder(nn.Module):
    """
    Exact Behavioral representation architecture
    reconstructed from the saved checkpoint.

    Input:
        87 processed features

    Representation:
        87
        ↓
        Linear(87 → 128)
        ↓
        BatchNorm1d(128)
        ↓
        ReLU
        ↓
        Dropout
        ↓
        Linear(128 → 128)
        ↓
        BatchNorm1d(128)
        ↓
        ReLU
        ↓
        128-D embedding

    The classifier is retained because it exists in the
    original checkpoint, but it is NOT used for generating
    the 128-D representation.
    """

    def __init__(self):

        super().__init__()

        self.encoder = nn.Sequential(

            # 87 → 128
            nn.Linear(
                87,
                128
            ),

            # Batch normalization
            nn.BatchNorm1d(
                128
            ),

            # Non-linear activation
            nn.ReLU(),

            # Dropout
            nn.Dropout(),

            # 128 → 128
            nn.Linear(
                128,
                128
            ),

            # Batch normalization
            nn.BatchNorm1d(
                128
            ),

            # Non-linear activation
            nn.ReLU(),
        )

        # Original classification head.
        # Not used for 128-D representation extraction.
        self.classifier = nn.Linear(
            128,
            1
        )

    def forward(self, x):

        embedding = self.encoder(x)

        return embedding


class EnvironmentalEncoder(nn.Module):
    """
    Exact Environmental representation architecture.

    Input:
        31 features

    Output:
        128-D embedding
    """

    def __init__(self):

        super().__init__()

        self.encoder = nn.Sequential(

            # 31 → 128
            nn.Linear(
                31,
                128
            ),

            nn.BatchNorm1d(
                128
            ),

            nn.ReLU(),

            nn.Dropout(
                0.15
            ),

            # 128 → 128
            nn.Linear(
                128,
                128
            ),

            nn.BatchNorm1d(
                128
            ),

            nn.ReLU(),
        )

        # Original classifier.
        # Not used for representation extraction.
        self.classifier = nn.Linear(
            128,
            1
        )

    def forward(self, x):

        embedding = self.encoder(x)

        return embedding


print("[PASSED] BehavioralEncoder defined.")
print("[PASSED] EnvironmentalEncoder defined.")

[PASSED] BehavioralEncoder defined.
[PASSED] EnvironmentalEncoder defined.


In [9]:
# ============================================================
# CELL 7 — VERIFY CHECKPOINT STRUCTURE
# ============================================================

print("=" * 70)
print("CHECKPOINT STRUCTURE VERIFICATION")
print("=" * 70)


# ------------------------------------------------------------
# Load Behavioral checkpoint
# ------------------------------------------------------------

behavioral_checkpoint = torch.load(
    BEHAVIORAL_MODEL_PATH,
    map_location=DEVICE,
    weights_only=True
)

behavioral_state = behavioral_checkpoint[
    "model_state_dict"
]


print("\nBehavioral checkpoint keys:")

for key in behavioral_state.keys():

    print(
        f"  {key:<45}"
        f"{tuple(behavioral_state[key].shape)}"
    )


# ------------------------------------------------------------
# Expected key structure
# ------------------------------------------------------------

expected_behavioral_keys = {
    "encoder.0.weight",
    "encoder.0.bias",
    "encoder.1.weight",
    "encoder.1.bias",
    "encoder.1.running_mean",
    "encoder.1.running_var",
    "encoder.1.num_batches_tracked",
    "encoder.4.weight",
    "encoder.4.bias",
    "encoder.5.weight",
    "encoder.5.bias",
    "encoder.5.running_mean",
    "encoder.5.running_var",
    "encoder.5.num_batches_tracked",
    "classifier.weight",
    "classifier.bias",
}


actual_behavioral_keys = set(
    behavioral_state.keys()
)


if actual_behavioral_keys != expected_behavioral_keys:

    missing = (
        expected_behavioral_keys
        - actual_behavioral_keys
    )

    unexpected = (
        actual_behavioral_keys
        - expected_behavioral_keys
    )

    raise ValueError(
        "\nBehavioral checkpoint structure mismatch.\n"
        f"Missing keys: {missing}\n"
        f"Unexpected keys: {unexpected}"
    )


print(
    "\n[PASSED] Behavioral checkpoint structure "
    "matches the reconstructed architecture."
)


# ------------------------------------------------------------
# Verify dimensions
# ------------------------------------------------------------

if tuple(
    behavioral_state[
        "encoder.0.weight"
    ].shape
) != (128, 87):

    raise ValueError(
        "Unexpected Behavioral first-layer shape."
    )


if tuple(
    behavioral_state[
        "encoder.4.weight"
    ].shape
) != (128, 128):

    raise ValueError(
        "Unexpected Behavioral second-layer shape."
    )


if tuple(
    behavioral_state[
        "classifier.weight"
    ].shape
) != (1, 128):

    raise ValueError(
        "Unexpected Behavioral classifier shape."
    )


print(
    "[PASSED] Behavioral dimensions verified."
)

CHECKPOINT STRUCTURE VERIFICATION

Behavioral checkpoint keys:
  encoder.0.weight                             (128, 87)
  encoder.0.bias                               (128,)
  encoder.1.weight                             (128,)
  encoder.1.bias                               (128,)
  encoder.1.running_mean                       (128,)
  encoder.1.running_var                        (128,)
  encoder.1.num_batches_tracked                ()
  encoder.4.weight                             (128, 128)
  encoder.4.bias                               (128,)
  encoder.5.weight                             (128,)
  encoder.5.bias                               (128,)
  encoder.5.running_mean                       (128,)
  encoder.5.running_var                        (128,)
  encoder.5.num_batches_tracked                ()
  classifier.weight                            (1, 128)
  classifier.bias                              (1,)

[PASSED] Behavioral checkpoint structure matches the reconstructed archit

In [11]:
# ============================================================
# CELL 8 — LOAD ALL FROZEN REPRESENTATION MODELS
# ============================================================

print("=" * 70)
print("LOADING FROZEN REPRESENTATION MODELS")
print("=" * 70)


# ============================================================
# 1. GENETIC REPRESENTATION MODEL
# ============================================================

genetic_state = torch.load(
    GENETIC_MODEL_PATH,
    map_location=DEVICE,
    weights_only=True
)

genetic_model = GeneticTransformer()

genetic_model.load_state_dict(
    genetic_state,
    strict=True
)

genetic_model.eval()

print(
    "[PASSED] Genetic representation model loaded."
)


# ============================================================
# 2. BEHAVIORAL REPRESENTATION MODEL
# ============================================================

behavioral_checkpoint = torch.load(
    BEHAVIORAL_MODEL_PATH,
    map_location=DEVICE,
    weights_only=True
)

behavioral_state = (
    behavioral_checkpoint[
        "model_state_dict"
    ]
)

behavioral_model = BehavioralEncoder()

behavioral_model.load_state_dict(
    behavioral_state,
    strict=True
)

behavioral_model.eval()

print(
    "[PASSED] Behavioral representation model loaded."
)


# ============================================================
# 3. ENVIRONMENTAL REPRESENTATION MODEL
# ============================================================
#
# IMPORTANT:
#
# This checkpoint contains additional metadata, including
# NumPy scalar objects.
#
# Therefore weights_only=True is not suitable for this
# particular trusted local checkpoint under the current
# PyTorch version.
#
# We use weights_only=False ONLY for this local artifact.
# ============================================================

environmental_checkpoint = torch.load(
    ENVIRONMENTAL_MODEL_PATH,
    map_location=DEVICE,
    weights_only=False
)

environmental_state = (
    environmental_checkpoint[
        "model_state_dict"
    ]
)

environmental_model = EnvironmentalEncoder()

environmental_model.load_state_dict(
    environmental_state,
    strict=True
)

environmental_model.eval()

print(
    "[PASSED] Environmental representation model loaded."
)


print("\n" + "=" * 70)
print("ALL REPRESENTATION MODELS LOADED SUCCESSFULLY")
print("=" * 70)

LOADING FROZEN REPRESENTATION MODELS
[PASSED] Genetic representation model loaded.
[PASSED] Behavioral representation model loaded.
[PASSED] Environmental representation model loaded.

ALL REPRESENTATION MODELS LOADED SUCCESSFULLY


In [16]:
# ============================================================
# CELL 9 — VERIFY CHECKPOINT STRUCTURES
# ============================================================

print("=" * 70)
print("CHECKPOINT STRUCTURE VERIFICATION")
print("=" * 70)


# ------------------------------------------------------------
# Genetic
# ------------------------------------------------------------

genetic_input_shape = genetic_state[
    "feature_embedding.weight"
].shape

genetic_output_shape = genetic_state[
    "output_projection.0.weight"
].shape


print("\nGENETIC")
print("Feature embedding shape :", genetic_input_shape)
print("Output projection shape :", genetic_output_shape)


# ------------------------------------------------------------
# Behavioral
# ------------------------------------------------------------

behavioral_input_shape = behavioral_state[
    "encoder.0.weight"
].shape

behavioral_second_shape = behavioral_state[
    "encoder.4.weight"
].shape

behavioral_classifier_shape = behavioral_state[
    "classifier.weight"
].shape


print("\nBEHAVIORAL")
print("First encoder shape     :", behavioral_input_shape)
print("Second encoder shape    :", behavioral_second_shape)
print("Classifier shape        :", behavioral_classifier_shape)


# ------------------------------------------------------------
# Environmental
# ------------------------------------------------------------

environmental_input_shape = environmental_state[
    "encoder.0.weight"
].shape

environmental_second_shape = environmental_state[
    "encoder.4.weight"
].shape

environmental_classifier_shape = environmental_state[
    "classifier.weight"
].shape


print("\nENVIRONMENTAL")
print("First encoder shape     :", environmental_input_shape)
print("Second encoder shape    :", environmental_second_shape)
print("Classifier shape        :", environmental_classifier_shape)


# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert genetic_input_shape == (128, 1)
assert genetic_output_shape == (128, 128)

assert behavioral_input_shape == (128, 87)
assert behavioral_second_shape == (128, 128)
assert behavioral_classifier_shape == (1, 128)

assert environmental_input_shape == (128, 31)
assert environmental_second_shape == (128, 128)
assert environmental_classifier_shape == (1, 128)


print("\n[PASSED] Genetic architecture verified.")
print("[PASSED] Behavioral architecture verified.")
print("[PASSED] Environmental architecture verified.")

print("\n[PASSED] All checkpoint dimensions are correct.")

CHECKPOINT STRUCTURE VERIFICATION

GENETIC
Feature embedding shape : torch.Size([128, 1])
Output projection shape : torch.Size([128, 128])

BEHAVIORAL
First encoder shape     : torch.Size([128, 87])
Second encoder shape    : torch.Size([128, 128])
Classifier shape        : torch.Size([1, 128])

ENVIRONMENTAL
First encoder shape     : torch.Size([128, 31])
Second encoder shape    : torch.Size([128, 128])
Classifier shape        : torch.Size([1, 128])

[PASSED] Genetic architecture verified.
[PASSED] Behavioral architecture verified.
[PASSED] Environmental architecture verified.

[PASSED] All checkpoint dimensions are correct.


In [17]:
# ============================================================
# CELL 10 — FREEZE ALL REPRESENTATION MODELS
# ============================================================

models = {
    "genetic": genetic_model,
    "behavioral": behavioral_model,
    "environmental": environmental_model,
}


for modality, model in models.items():

    model.eval()

    for parameter in model.parameters():
        parameter.requires_grad = False


print("=" * 70)
print("MODEL FREEZE VERIFICATION")
print("=" * 70)


for modality, model in models.items():

    trainable_count = 0
    total_parameter_count = 0

    for parameter in model.parameters():

        total_parameter_count += parameter.numel()

        if parameter.requires_grad:
            trainable_count += parameter.numel()

    print(
        f"{modality.capitalize():<15}"
        f" total parameters = {total_parameter_count:,}"
        f" | trainable = {trainable_count:,}"
    )

    assert trainable_count == 0


print("\n[PASSED] All representation models are frozen.")
print("[PASSED] All models are in evaluation mode.")

MODEL FREEZE VERIFICATION
Genetic         total parameters = 281,984 | trainable = 0
Behavioral      total parameters = 28,417 | trainable = 0
Environmental   total parameters = 21,249 | trainable = 0

[PASSED] All representation models are frozen.
[PASSED] All models are in evaluation mode.


In [18]:
# ============================================================
# CELL 11 — FORWARD-PASS SANITY TEST
# ============================================================

print("=" * 70)
print("FORWARD-PASS SANITY TEST")
print("=" * 70)


with torch.inference_mode():

    # --------------------------------------------------------
    # Genetic
    # --------------------------------------------------------

    genetic_dummy = torch.zeros(
        2,
        1000,
        dtype=torch.float32,
        device=DEVICE
    )

    genetic_result = genetic_model(
        genetic_dummy
    )


    # --------------------------------------------------------
    # Behavioral
    # --------------------------------------------------------

    behavioral_dummy = torch.zeros(
        2,
        87,
        dtype=torch.float32,
        device=DEVICE
    )

    behavioral_result = behavioral_model(
        behavioral_dummy
    )


    # --------------------------------------------------------
    # Environmental
    # --------------------------------------------------------

    environmental_dummy = torch.zeros(
        2,
        31,
        dtype=torch.float32,
        device=DEVICE
    )

    environmental_result = environmental_model(
        environmental_dummy
    )


print("\nGenetic")
print("Input shape :", genetic_dummy.shape)
print("Output shape:", genetic_result.shape)

print("\nBehavioral")
print("Input shape :", behavioral_dummy.shape)
print("Output shape:", behavioral_result.shape)

print("\nEnvironmental")
print("Input shape :", environmental_dummy.shape)
print("Output shape:", environmental_result.shape)


assert genetic_result.shape == (2, 128)
assert behavioral_result.shape == (2, 128)
assert environmental_result.shape == (2, 128)


print("\n[PASSED] Genetic 1000-D → 128-D")
print("[PASSED] Behavioral 87-D → 128-D")
print("[PASSED] Environmental 31-D → 128-D")

print("\n[PASSED] All forward passes are valid.")

FORWARD-PASS SANITY TEST

Genetic
Input shape : torch.Size([2, 1000])
Output shape: torch.Size([2, 128])

Behavioral
Input shape : torch.Size([2, 87])
Output shape: torch.Size([2, 128])

Environmental
Input shape : torch.Size([2, 31])
Output shape: torch.Size([2, 128])

[PASSED] Genetic 1000-D → 128-D
[PASSED] Behavioral 87-D → 128-D
[PASSED] Environmental 31-D → 128-D

[PASSED] All forward passes are valid.


In [19]:
# ============================================================
# CELL 12 — GENETIC INPUT PREPARATION
# ============================================================

def prepare_genetic_input(csv_path):

    df = pd.read_csv(csv_path)

    print("Genetic file:", csv_path)
    print("Raw shape  :", df.shape)


    excluded_columns = {
        "GSM_ID",
        "Target",
        "Class/ASD",
        "Sample_Index",
    }


    feature_columns = []

    for column in df.columns:

        if column not in excluded_columns:
            feature_columns.append(column)


    numeric_features = df[
        feature_columns
    ].select_dtypes(
        include=[np.number]
    )


    if numeric_features.shape[1] != 1000:

        raise ValueError(
            "Expected exactly 1000 Genetic "
            "numeric features, but found "
            + str(numeric_features.shape[1])
        )


    X = numeric_features.to_numpy(
        dtype=np.float32
    )


    if not np.isfinite(X).all():

        raise ValueError(
            "Genetic input contains NaN or infinite values."
        )


    return X


print("[PASSED] Genetic input function defined.")

[PASSED] Genetic input function defined.


In [20]:
# ============================================================
# CELL 13 — GENETIC 128-D INFERENCE
# ============================================================

def run_genetic_inference(csv_path):

    X = prepare_genetic_input(
        csv_path
    )


    X_tensor = torch.tensor(
        X,
        dtype=torch.float32,
        device=DEVICE
    )


    with torch.inference_mode():

        embedding = genetic_model(
            X_tensor
        )


    embedding = embedding.cpu().numpy()


    if embedding.ndim != 2:

        raise ValueError(
            "Invalid Genetic embedding dimensions: "
            + str(embedding.shape)
        )


    if embedding.shape[1] != 128:

        raise ValueError(
            "Expected 128-D Genetic embedding, "
            "but received "
            + str(embedding.shape)
        )


    if not np.isfinite(embedding).all():

        raise ValueError(
            "Genetic embedding contains NaN or infinity."
        )


    return embedding


genetic_embeddings = {}


for patient_id in DEMO_PATIENTS:

    print("\nProcessing:", patient_id)

    embedding = run_genetic_inference(
        demo_files[patient_id]["genetic"]
    )

    genetic_embeddings[patient_id] = embedding[0]

    print(
        "Final embedding shape:",
        embedding.shape
    )


print("\n[PASSED] Genetic inference completed.")


Processing: P001
Genetic file: C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\05_multimodal_fusion\M4_demo_patients\P001\genetic_input.csv
Raw shape  : (1, 1003)
Final embedding shape: (1, 128)

Processing: P002
Genetic file: C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\05_multimodal_fusion\M4_demo_patients\P002\genetic_input.csv
Raw shape  : (1, 1003)
Final embedding shape: (1, 128)

Processing: P003
Genetic file: C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\05_multimodal_fusion\M4_demo_patients\P003\genetic_input.csv
Raw shape  : (1, 1003)
Final embedding shape: (1, 128)

Processing: P004
Genetic file: C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\05_multimodal_fusion\M4_demo_patients\P004\genetic_input.csv
Raw shape  : (1, 1003)
Final embedding shape: (1, 128)

Processing: P005
Genetic file: C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\05_multimodal_fusion\M4_de

In [21]:
# ============================================================
# CELL 14 — BEHAVIORAL RAW FEATURE DEFINITION
# ============================================================

BEHAVIORAL_RAW_FEATURES = [
    "A1_Score",
    "A2_Score",
    "A3_Score",
    "A4_Score",
    "A5_Score",
    "A6_Score",
    "A7_Score",
    "A8_Score",
    "A9_Score",
    "A10_Score",
    "age",
    "gender",
    "ethnicity",
    "jundice",
    "austim",
    "contry_of_res",
    "used_app_before",
    "relation",
]


print(
    "Behavioral raw feature count:",
    len(BEHAVIORAL_RAW_FEATURES)
)


assert len(BEHAVIORAL_RAW_FEATURES) == 18


print("[PASSED] Exactly 18 raw Behavioral features.")

Behavioral raw feature count: 18
[PASSED] Exactly 18 raw Behavioral features.


In [22]:
# ============================================================
# CELL 15 — BEHAVIORAL COMPATIBILITY PREPROCESSING
# ============================================================

def preprocess_behavioral_compatibility(
    dataframe
):

    missing_features = []

    for column in BEHAVIORAL_RAW_FEATURES:

        if column not in dataframe.columns:
            missing_features.append(column)


    if len(missing_features) > 0:

        raise ValueError(
            "Missing Behavioral features: "
            + str(missing_features)
        )


    production_df = dataframe[
        BEHAVIORAL_RAW_FEATURES
    ].copy()


    with tempfile.TemporaryDirectory() as temp_dir:

        temp_dir = Path(temp_dir)

        input_csv = (
            temp_dir
            / "behavioral_input.csv"
        )

        output_npy = (
            temp_dir
            / "behavioral_processed.npy"
        )


        production_df.to_csv(
            input_csv,
            index=False
        )


        helper_code = r"""
import sys
import numpy as np
import pandas as pd
import joblib

input_csv = sys.argv[1]
preprocessor_path = sys.argv[2]
output_npy = sys.argv[3]

data = pd.read_csv(input_csv)

preprocessor = joblib.load(
    preprocessor_path
)

processed = preprocessor.transform(
    data
)

processed = np.asarray(
    processed,
    dtype=np.float32
)

np.save(
    output_npy,
    processed
)

print(
    "PROCESSED_SHAPE=" + str(processed.shape)
)
"""


        result = subprocess.run(
            [
                str(BEHAVIORAL_PYTHON),
                "-c",
                helper_code,
                str(input_csv),
                str(BEHAVIORAL_PREPROCESSOR_PATH),
                str(output_npy),
            ],
            capture_output=True,
            text=True,
            check=True,
        )


        print(
            result.stdout.strip()
        )


        processed = np.load(
            output_npy
        )


    if processed.ndim != 2:

        raise ValueError(
            "Behavioral preprocessing returned "
            "invalid shape: "
            + str(processed.shape)
        )


    if processed.shape[1] != 87:

        raise ValueError(
            "Expected 87 processed Behavioral "
            "features, received "
            + str(processed.shape[1])
        )


    if not np.isfinite(processed).all():

        raise ValueError(
            "Behavioral preprocessing generated "
            "NaN or infinity."
        )


    return processed


print(
    "[PASSED] Behavioral compatibility "
    "preprocessor ready."
)

[PASSED] Behavioral compatibility preprocessor ready.


In [23]:
# ============================================================
# CELL 16 — BEHAVIORAL 128-D INFERENCE
# ============================================================

def run_behavioral_inference(csv_path):

    raw_data = pd.read_csv(
        csv_path
    )


    print(
        "Raw Behavioral shape:",
        raw_data.shape
    )


    processed = preprocess_behavioral_compatibility(
        raw_data
    )


    print(
        "Processed Behavioral shape:",
        processed.shape
    )


    processed_tensor = torch.tensor(
        processed,
        dtype=torch.float32,
        device=DEVICE
    )


    with torch.inference_mode():

        embedding = behavioral_model(
            processed_tensor
        )


    embedding = embedding.cpu().numpy()


    if embedding.shape[1] != 128:

        raise ValueError(
            "Expected Behavioral 128-D embedding, "
            "received "
            + str(embedding.shape)
        )


    if not np.isfinite(embedding).all():

        raise ValueError(
            "Behavioral embedding contains "
            "NaN or infinity."
        )


    return embedding


behavioral_embeddings = {}


for patient_id in DEMO_PATIENTS:

    print("\nProcessing:", patient_id)

    embedding = run_behavioral_inference(
        demo_files[patient_id]["behavioral"]
    )

    behavioral_embeddings[patient_id] = embedding[0]

    print(
        "Final embedding shape:",
        embedding.shape
    )


print(
    "\n[PASSED] Behavioral inference completed."
)


Processing: P001
Raw Behavioral shape: (1, 19)
PROCESSED_SHAPE=(1, 87)
Processed Behavioral shape: (1, 87)
Final embedding shape: (1, 128)

Processing: P002
Raw Behavioral shape: (1, 19)
PROCESSED_SHAPE=(1, 87)
Processed Behavioral shape: (1, 87)
Final embedding shape: (1, 128)

Processing: P003
Raw Behavioral shape: (1, 19)
PROCESSED_SHAPE=(1, 87)
Processed Behavioral shape: (1, 87)
Final embedding shape: (1, 128)

Processing: P004
Raw Behavioral shape: (1, 19)
PROCESSED_SHAPE=(1, 87)
Processed Behavioral shape: (1, 87)
Final embedding shape: (1, 128)

Processing: P005
Raw Behavioral shape: (1, 19)
PROCESSED_SHAPE=(1, 87)
Processed Behavioral shape: (1, 87)
Final embedding shape: (1, 128)

[PASSED] Behavioral inference completed.


In [24]:
# ============================================================
# CELL 17 — ENVIRONMENTAL PREPROCESSOR
# ============================================================

environmental_preprocessor = joblib.load(
    ENVIRONMENTAL_PREPROCESSOR_PATH
)


print("=" * 70)
print("ENVIRONMENTAL PREPROCESSOR")
print("=" * 70)

print(
    "Type:",
    type(environmental_preprocessor)
)

print(
    "Input features:",
    environmental_preprocessor.n_features_in_
)


assert (
    environmental_preprocessor.n_features_in_
    == 31
)


print(
    "\n[PASSED] Environmental preprocessor "
    "expects exactly 31 features."
)

ENVIRONMENTAL PREPROCESSOR
Type: <class 'sklearn.pipeline.Pipeline'>
Input features: 31

[PASSED] Environmental preprocessor expects exactly 31 features.


In [25]:
# ============================================================
# CELL 18 — ENVIRONMENTAL 128-D INFERENCE
# ============================================================

def run_environmental_inference(csv_path):

    raw_data = pd.read_csv(
        csv_path
    )


    print(
        "Raw Environmental shape:",
        raw_data.shape
    )


    feature_columns = (
        environmental_checkpoint[
            "feature_columns"
        ]
    )


    if len(feature_columns) != 31:

        raise ValueError(
            "Environmental checkpoint does not "
            "contain exactly 31 feature columns."
        )


    missing_features = []

    for column in feature_columns:

        if column not in raw_data.columns:
            missing_features.append(column)


    if len(missing_features) > 0:

        raise ValueError(
            "Missing Environmental features: "
            + str(missing_features)
        )


    X_raw = raw_data[
        feature_columns
    ].copy()


    X_processed = (
        environmental_preprocessor.transform(
            X_raw
        )
    )


    X_processed = np.asarray(
        X_processed,
        dtype=np.float32
    )


    print(
        "Processed Environmental shape:",
        X_processed.shape
    )


    if X_processed.shape[1] != 31:

        raise ValueError(
            "Expected 31 processed Environmental "
            "features, received "
            + str(X_processed.shape[1])
        )


    if not np.isfinite(X_processed).all():

        raise ValueError(
            "Environmental preprocessing generated "
            "NaN or infinity."
        )


    X_tensor = torch.tensor(
        X_processed,
        dtype=torch.float32,
        device=DEVICE
    )


    with torch.inference_mode():

        embedding = environmental_model(
            X_tensor
        )


    embedding = embedding.cpu().numpy()


    if embedding.shape[1] != 128:

        raise ValueError(
            "Expected Environmental 128-D embedding, "
            "received "
            + str(embedding.shape)
        )


    if not np.isfinite(embedding).all():

        raise ValueError(
            "Environmental embedding contains "
            "NaN or infinity."
        )


    return embedding


environmental_embeddings = {}


for patient_id in DEMO_PATIENTS:

    print("\nProcessing:", patient_id)

    embedding = run_environmental_inference(
        demo_files[patient_id]["environmental"]
    )

    environmental_embeddings[patient_id] = embedding[0]

    print(
        "Final embedding shape:",
        embedding.shape
    )


print(
    "\n[PASSED] Environmental inference completed."
)


Processing: P001
Raw Environmental shape: (1, 32)
Processed Environmental shape: (1, 31)
Final embedding shape: (1, 128)

Processing: P002
Raw Environmental shape: (1, 32)
Processed Environmental shape: (1, 31)
Final embedding shape: (1, 128)

Processing: P003
Raw Environmental shape: (1, 32)
Processed Environmental shape: (1, 31)
Final embedding shape: (1, 128)

Processing: P004
Raw Environmental shape: (1, 32)
Processed Environmental shape: (1, 31)
Final embedding shape: (1, 128)

Processing: P005
Raw Environmental shape: (1, 32)
Processed Environmental shape: (1, 31)
Final embedding shape: (1, 128)

[PASSED] Environmental inference completed.


In [26]:
# ============================================================
# CELL 19 — VERIFY ALL 128-D EMBEDDINGS
# ============================================================

print("=" * 70)
print("FINAL 128-D EMBEDDING VERIFICATION")
print("=" * 70)


all_embeddings = {
    "Genetic": genetic_embeddings,
    "Behavioral": behavioral_embeddings,
    "Environmental": environmental_embeddings,
}


for modality in all_embeddings:

    print("\n" + modality)

    embeddings = all_embeddings[
        modality
    ]


    assert len(embeddings) == 5


    for patient_id in DEMO_PATIENTS:

        vector = np.asarray(
            embeddings[patient_id]
        )


        print(
            " ",
            patient_id,
            "→",
            vector.shape
        )


        assert vector.shape == (128,)


        assert np.isfinite(
            vector
        ).all()


print("\n[PASSED] Genetic: 5 × 128-D")
print("[PASSED] Behavioral: 5 × 128-D")
print("[PASSED] Environmental: 5 × 128-D")

print(
    "\n[PASSED] All 15 modality embeddings "
    "are valid."
)

FINAL 128-D EMBEDDING VERIFICATION

Genetic
  P001 → (128,)
  P002 → (128,)
  P003 → (128,)
  P004 → (128,)
  P005 → (128,)

Behavioral
  P001 → (128,)
  P002 → (128,)
  P003 → (128,)
  P004 → (128,)
  P005 → (128,)

Environmental
  P001 → (128,)
  P002 → (128,)
  P003 → (128,)
  P004 → (128,)
  P005 → (128,)

[PASSED] Genetic: 5 × 128-D
[PASSED] Behavioral: 5 × 128-D
[PASSED] Environmental: 5 × 128-D

[PASSED] All 15 modality embeddings are valid.


In [27]:
# ============================================================
# CELL 20 — CREATE 128-D EMBEDDING TABLES
# ============================================================

def create_embedding_dataframe(
    embedding_dictionary,
    prefix
):

    rows = []


    for patient_id in DEMO_PATIENTS:

        vector = np.asarray(
            embedding_dictionary[
                patient_id
            ]
        )


        row = {
            "patient_id": patient_id
        }


        for index in range(128):

            column_name = (
                prefix
                + "_128D_"
                + str(index + 1).zfill(3)
            )

            row[column_name] = float(
                vector[index]
            )


        rows.append(row)


    return pd.DataFrame(rows)


genetic_128_df = create_embedding_dataframe(
    genetic_embeddings,
    "Genetic"
)

behavioral_128_df = create_embedding_dataframe(
    behavioral_embeddings,
    "Behavioral"
)

environmental_128_df = create_embedding_dataframe(
    environmental_embeddings,
    "Environmental"
)


print(
    "Genetic table shape:",
    genetic_128_df.shape
)

print(
    "Behavioral table shape:",
    behavioral_128_df.shape
)

print(
    "Environmental table shape:",
    environmental_128_df.shape
)

Genetic table shape: (5, 129)
Behavioral table shape: (5, 129)
Environmental table shape: (5, 129)


In [28]:
# ============================================================
# CELL 21 — SAVE 05.02 RESULTS
# ============================================================

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


genetic_output = (
    REPORT_DIR
    / "05_02_genetic_128d_embeddings.csv"
)

behavioral_output = (
    REPORT_DIR
    / "05_02_behavioral_128d_embeddings.csv"
)

environmental_output = (
    REPORT_DIR
    / "05_02_environmental_128d_embeddings.csv"
)


genetic_128_df.to_csv(
    genetic_output,
    index=False
)

behavioral_128_df.to_csv(
    behavioral_output,
    index=False
)

environmental_128_df.to_csv(
    environmental_output,
    index=False
)


print("=" * 70)
print("05.02 RESULTS SAVED")
print("=" * 70)

print(
    "\nGenetic:",
    genetic_output
)

print(
    "Behavioral:",
    behavioral_output
)

print(
    "Environmental:",
    environmental_output
)

05.02 RESULTS SAVED

Genetic: C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\05_multimodal_integration\reports\05_02_modality_inference\05_02_genetic_128d_embeddings.csv
Behavioral: C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\05_multimodal_integration\reports\05_02_modality_inference\05_02_behavioral_128d_embeddings.csv
Environmental: C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\05_multimodal_integration\reports\05_02_modality_inference\05_02_environmental_128d_embeddings.csv


In [29]:
# ============================================================
# CELL 22 — CREATE 05.02 REPORT
# ============================================================

report_data = {

    "phase": "05_multimodal_integration",

    "notebook": "05_02_modality_inference",

    "objective": (
        "Generate frozen 128-D "
        "modality-specific representations."
    ),

    "training_performed": False,

    "checkpoint_modified": False,

    "projection_128_to_32_performed": False,

    "cross_modal_inference_performed": False,

    "multimodal_fusion_performed": False,

    "demo_patients": DEMO_PATIENTS,

    "modalities": {

        "Genetic": {
            "input_dimension": 1000,
            "output_dimension": 128,
            "model": "GeneticTransformer",
        },

        "Behavioral": {
            "raw_dimension": 18,
            "processed_dimension": 87,
            "output_dimension": 128,
            "model": "BehavioralEncoder",
            "sklearn_compatibility": "1.6.1",
        },

        "Environmental": {
            "input_dimension": 31,
            "output_dimension": 128,
            "model": "EnvironmentalEncoder",
        },
    },

    "outputs": {
        "genetic": str(genetic_output),
        "behavioral": str(behavioral_output),
        "environmental": str(environmental_output),
    },
}


report_path = (
    REPORT_DIR
    / "05_02_modality_inference_report.json"
)


with open(
    report_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        report_data,
        file,
        indent=4
    )


print(
    "[PASSED] Report saved:"
)

print(
    report_path
)

[PASSED] Report saved:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\05_multimodal_integration\reports\05_02_modality_inference\05_02_modality_inference_report.json


In [30]:
# ============================================================
# CELL 23 — FINAL PHASE 05.02 VALIDATION
# ============================================================

print("=" * 70)
print("PHASE 05.02 FINAL VALIDATION")
print("=" * 70)


required_outputs = [
    genetic_output,
    behavioral_output,
    environmental_output,
    report_path,
]


for output_file in required_outputs:

    if not output_file.exists():

        raise FileNotFoundError(
            "Missing expected output: "
            + str(output_file)
        )

    print(
        "[PASSED]",
        output_file.name
    )


# ------------------------------------------------------------
# Final dimensional validation
# ------------------------------------------------------------

assert genetic_128_df.shape == (5, 129)

assert behavioral_128_df.shape == (5, 129)

assert environmental_128_df.shape == (5, 129)


print("\n[PASSED] Genetic output: 5 × 128-D")
print("[PASSED] Behavioral output: 5 × 128-D")
print("[PASSED] Environmental output: 5 × 128-D")


print("\n" + "=" * 70)
print("PHASE 05.02 STATUS: PASSED")
print("=" * 70)


print(
    """
No training performed.
No checkpoint modified.

Genetic:
    1000-D → 128-D

Behavioral:
    18-D raw → 87-D processed → 128-D

Environmental:
    31-D → 128-D

Next phase:
    05.03 Common-Dimensional Projection
"""
)

PHASE 05.02 FINAL VALIDATION
[PASSED] 05_02_genetic_128d_embeddings.csv
[PASSED] 05_02_behavioral_128d_embeddings.csv
[PASSED] 05_02_environmental_128d_embeddings.csv
[PASSED] 05_02_modality_inference_report.json

[PASSED] Genetic output: 5 × 128-D
[PASSED] Behavioral output: 5 × 128-D
[PASSED] Environmental output: 5 × 128-D

PHASE 05.02 STATUS: PASSED

No training performed.
No checkpoint modified.

Genetic:
    1000-D → 128-D

Behavioral:
    18-D raw → 87-D processed → 128-D

Environmental:
    31-D → 128-D

Next phase:
    05.03 Common-Dimensional Projection

